In [ ]:
# --- Configure paths and run ZeoSyn evaluation ---
# Set to your actual ZeoSyn file path (Excel or CSV with columns: doi, Code1, Code2, Code3)
ZEOSYN_PATH = "ZEOSYN.xlsx" 
QUESTIONS_PATH = "zeolite_synthesis_questions.txt"  # provided

# Load ZeoSyn mapping
try:
    zeo_map = load_zeosyn_framework_to_doi_map(ZEOSYN_PATH)
    print(f"Loaded ZeoSyn map with {len(zeo_map)} framework entries")
except Exception as e:
    print("Failed to load ZeoSyn file:", e)
    zeo_map = {}

# Evaluate
summary, per_q = evaluate_questions_against_zeosyn(
    QUESTIONS_PATH,
    vector_db,
    zeo_map,
    ks=(1,3,5)
)

print("ZeoSyn Retrieval Accuracy (Hit@K by framework):")
for k, v in summary["accuracies"].items():
    print(f"  {k}: {v:.2f}%  (hits={summary['hits'][int(k.split('_')[1])]}/total={summary['totals'][int(k.split('_')[1])]})")

# Save detailed results
details_path_json = "zeosyn_benchmark_details.json"
with open(details_path_json, "w") as f:
    json.dump({"summary": summary, "results": per_q}, f, indent=2)
print(f"Saved detailed results to {details_path_json}")

In [ ]:
import matplotlib.pyplot as plt

labels = ["Top-1", "Top-3", "Top-5"]
vals = [summary["accuracies"]["top_1"], summary["accuracies"]["top_3"], summary["accuracies"]["top_5"]]
plt.figure(figsize=(8,5))
plt.bar(labels, vals, color=["#2ecc71", "#3498db", "#9b59b6"]) 
plt.ylim(0, 100)
for i, v in enumerate(vals):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center")
plt.title("ZeoSyn Hit@K accuracy (framework-aware)")
plt.ylabel("Accuracy (%)")
plt.show()